# Test Data Setup

Maak testdata aan in **Salesforce** en **SmartSales** voor de entity-centric evaluatieprompts.

Elke sectie = één entiteit. Voer de cellen per entiteit uit van boven naar beneden — de `account_id` van een vorige cel wordt automatisch doorgegeven aan de volgende.

## 0. Setup & Auth

Eénmalig uitvoeren. Haalt tokens op voor beide systemen via de bestaande auth modules.

In [45]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import httpx
from dotenv import load_dotenv
load_dotenv()

# ── Salesforce ─────────────────────────────────────────────────────────────
from salesforce.auth import authenticate_salesforce
sf_creds = authenticate_salesforce('https://test.salesforce.com')
SF_TOKEN = sf_creds.access_token
SF_BASE  = sf_creds.instance_url.rstrip('/')
SF_API   = f'{SF_BASE}/services/data/v59.0'
print(f'✓ Salesforce  instance={SF_BASE}')

# ── SmartSales ─────────────────────────────────────────────────────────────
from smartsales.auth import authenticate_from_env
ss_creds = authenticate_from_env()
SS_TOKEN = ss_creds.access_token
SS_BASE  = 'https://proxy-smartsales.easi.net/proxy/rest'
print(f'✓ SmartSales  token_ok=True')

✓ Salesforce  instance=https://demoeasi2023--aiseach.sandbox.my.salesforce.com
✓ SmartSales  token_ok=True


In [46]:

# ── Helper functies ────────────────────────────────────────────────────────

def sf_post(obj_type: str, payload: dict) -> dict:
    """Maak een Salesforce object aan.
    Bij een duplicate-waarschuwing: geeft het bestaande record terug (idempotent).
    """
    url = f'{SF_API}/sobjects/{obj_type}/'
    r = httpx.post(url, json=payload, headers={'Authorization': f'Bearer {SF_TOKEN}'})
    if r.status_code == 400:
        try:
            errors = r.json()
            for err in errors:
                if err.get('errorCode') == 'DUPLICATES_DETECTED':
                    match_records = (
                        err.get('duplicateResult', {})
                           .get('matchResults', [{}])[0]
                           .get('matchRecords', [{}])
                    )
                    if match_records:
                        existing_id = match_records[0]['record']['Id']
                        print(f'↩ {obj_type} bestaat al — hergebruik  id={existing_id}')
                        return {'id': existing_id, 'success': True, 'reused': True}
        except Exception:
            pass
        print(f'❌ Salesforce fout [{r.status_code}]: {r.text}')
        r.raise_for_status()
    r.raise_for_status()
    return r.json()

def sf_query(soql: str) -> list:
    url = f'{SF_API}/query'
    r = httpx.get(url, params={'q': soql}, headers={'Authorization': f'Bearer {SF_TOKEN}'})
    r.raise_for_status()
    return r.json().get('records', [])

def sf_get_account_id(name: str) -> str | None:
    recs = sf_query(f"SELECT Id FROM Account WHERE Name = '{name}' LIMIT 1")
    return recs[0]['Id'] if recs else None

def ss_post(payload: dict) -> dict:
    url = f'{SS_BASE}/api/v3/location'
    r = httpx.post(url, json=payload, headers={'Authorization': f'Bearer {SS_TOKEN}'})
    if not r.is_success:
        print(f'❌ SmartSales fout [{r.status_code}]: {r.text}')
        r.raise_for_status()
    return r.json()

def ss_search(name: str) -> list:
    import json as _json
    url = f'{SS_BASE}/api/v3/location/list'
    q = _json.dumps({'name': f'contains:{name}'})
    r = httpx.get(url, params={'q': q, 'p': 'simple'}, headers={'Authorization': f'Bearer {SS_TOKEN}'})
    r.raise_for_status()
    return r.json().get('entries', [])

def ss_get_or_create(payload: dict) -> dict:
    """Zoek een locatie op exacte naam. Maak aan als niet gevonden."""
    import json as _json
    name = payload['name']
    url = f'{SS_BASE}/api/v3/location/list'
    q = _json.dumps({'name': f'eq:{name}'})
    r = httpx.get(url, params={'q': q, 'p': 'simple'}, headers={'Authorization': f'Bearer {SS_TOKEN}'})
    r.raise_for_status()
    entries = r.json().get('entries', [])
    if entries:
        loc = entries[0]
        print(f'↩ Locatie "{name}" bestaat al  uid={loc.get("uid")}')
        return loc
    result = ss_post(payload)
    print(f'✓ Locatie "{name}" aangemaakt  uid={result.get("uid")}')
    return result

print('✓ helpers geladen')


✓ helpers geladen


---
## 1. supplier1

Nodig voor prompts: *Tell me everything about supplier1*, *current status of supplier1*, *files/meetings/deals linked to supplier1*, *how active have we been with supplier1*

In [47]:

# 1a. Salesforce — Account (hergebruik als al bestaat)
existing = sf_get_account_id('supplier1')
if existing:
    SUPPLIER1_ID = existing
    print(f'Account supplier1 bestaat al  id={SUPPLIER1_ID}')
else:
    res = sf_post('Account', {
        'Name': 'supplier1',
        'Industry': 'Manufacturing',
        'BillingCity': 'Brussels',
        'BillingCountry': 'Belgium',
        'Phone': '+32 2 123 45 67',
        'Description': 'Main test supplier for evaluation scenarios.',
    })
    SUPPLIER1_ID = res['id']
    print(f'Account supplier1 aangemaakt  id={SUPPLIER1_ID}')


Account supplier1 bestaat al  id=001KI00000N7MjVYAV


In [48]:
# 1b. Salesforce — Contact gekoppeld aan supplier1
res = sf_post('Contact', {
    'FirstName': 'Marc',
    'LastName': 'Supplier',
    'Email': 'marc.supplier@supplier1.be',
    'Title': 'Account Manager',
    'AccountId': SUPPLIER1_ID,
})
print(f'Contact Marc Supplier  id={res["id"]}')

↩ Contact bestaat al — hergebruik  id=003KI000009uzQsYAI
Contact Marc Supplier  id=003KI000009uzQsYAI


In [49]:
# 1c. Salesforce — Opportunity gekoppeld aan supplier1
res = sf_post('Opportunity', {
    'Name': 'supplier1 — Q2 Delivery Contract',
    'StageName': 'Negotiation/Review',
    'CloseDate': '2026-06-30',
    'Amount': 85000,
    'AccountId': SUPPLIER1_ID,
    'Description': 'Renewal of annual delivery agreement for Q2 2026.',
})
print(f'Opportunity  id={res["id"]}')

Opportunity  id=006KI000005bHekYAE


In [13]:
# 1d. Salesforce — open Case gekoppeld aan supplier1
res = sf_post('Case', {
    'Subject': 'Delayed shipment — supplier1 April batch',
    'Status': 'Working',
    'Priority': 'High',
    'AccountId': SUPPLIER1_ID,
    'Description': 'The April delivery batch arrived 5 days late. Investigating root cause.',
})
print(f'Case  id={res["id"]}')

Case  id=500KI000001mJjsYAE


In [17]:

# 1e. SmartSales — Locatie voor supplier1 (hergebruik als al bestaat)
res = ss_get_or_create({
    'name': 'supplier1',
    'code': 'SUP-001',
    'city': 'Brussels',
    'country': 'Belgium',
    'street': 'Rue du Commerce 1',
    'zip': '1000',
})


Locatie "supplier1" bestaat al  uid=cac5b20d-285c-11f1-a2fe-005056010707


---
## 2. Colruyt

Nodig voor prompt: *Give me a full briefing on Colruyt*

In [18]:

# 2a. Salesforce — Account (hergebruik als al bestaat)
existing = sf_get_account_id('Colruyt Group')
if existing:
    COLRUYT_ID = existing
    print(f'Account Colruyt Group bestaat al  id={COLRUYT_ID}')
else:
    res = sf_post('Account', {
        'Name': 'Colruyt Group',
        'Industry': 'Retail',
        'BillingCity': 'Halle',
        'BillingCountry': 'Belgium',
        'Phone': '+32 2 360 10 40',
        'Website': 'https://www.colruytgroup.com',
    })
    COLRUYT_ID = res['id']
    print(f'Account Colruyt Group aangemaakt  id={COLRUYT_ID}')


Account Colruyt Group aangemaakt  id=001KI00000N7zHSYAZ


In [19]:
# 2b. Salesforce — Opportunity
res = sf_post('Opportunity', {
    'Name': 'Colruyt Group — SmartSales Rollout 2026',
    'StageName': 'Proposal/Price Quote',
    'CloseDate': '2026-09-30',
    'Amount': 220000,
    'AccountId': COLRUYT_ID,
})
print(f'Opportunity  id={res["id"]}')

Opportunity  id=006KI000005bHl1YAE


In [20]:
# 2c. Salesforce — Case
res = sf_post('Case', {
    'Subject': 'Colruyt — Integration issue with POS system',
    'Status': 'New',
    'Priority': 'Medium',
    'AccountId': COLRUYT_ID,
})
print(f'Case  id={res["id"]}')

Case  id=500KI000001mJiCYAU


In [21]:

# 2d. SmartSales — Locatie Colruyt
res = ss_get_or_create({
    'name': 'Colruyt - Halle',
    'code': 'COLRUYT-HAL-001',
    'city': 'Halle',
    'country': 'Belgium',
    'street': 'Edingensesteenweg 196',
    'zip': '1500',
})


Locatie "Colruyt - Halle" aangemaakt  uid=329a7e6c-42da-11f1-a803-005056010707


---
## 3. GreenTech Solutions

Nodig voor prompt: *What do we know about GreenTech Solutions?*

In [22]:

# 3a. Salesforce — Account (hergebruik als al bestaat)
existing = sf_get_account_id('GreenTech Solutions')
if existing:
    GREENTECH_ID = existing
    print(f'Account GreenTech Solutions bestaat al  id={GREENTECH_ID}')
else:
    res = sf_post('Account', {
        'Name': 'GreenTech Solutions',
        'Industry': 'Technology',
        'BillingCity': 'Ghent',
        'BillingCountry': 'Belgium',
        'Website': 'https://www.greentechsolutions.be',
    })
    GREENTECH_ID = res['id']
    print(f'Account GreenTech Solutions aangemaakt  id={GREENTECH_ID}')


Account GreenTech Solutions aangemaakt  id=001KI00000N7zqXYAR


In [23]:
# 3b. Salesforce — Opportunity
res = sf_post('Opportunity', {
    'Name': 'GreenTech Solutions — Sustainability Dashboard',
    'StageName': 'Qualification',
    'CloseDate': '2026-08-15',
    'Amount': 45000,
    'AccountId': GREENTECH_ID,
})
print(f'Opportunity  id={res["id"]}')

Opportunity  id=006KI000005bHl6YAE


In [24]:

# 3c. SmartSales — Locatie GreenTech
res = ss_get_or_create({
    'name': 'GreenTech Solutions - Ghent',
    'code': 'GTS-GNT-001',
    'city': 'Ghent',
    'country': 'Belgium',
    'street': 'Technologiepark 122',
    'zip': '9052',
})


Locatie "GreenTech Solutions - Ghent" aangemaakt  uid=3d28b72d-42da-11f1-a803-005056010707


---
## 4. Dorian (persoon)

Nodig voor prompt: *Who is Dorian and how do we work with him?*

> Pas `DORIAN_LAST` en `DORIAN_EMAIL` aan naar zijn echte achternaam/email als je die kent uit je bestaande emails.

In [25]:
DORIAN_LAST  = 'Feaux'          # ← pas aan indien nodig
DORIAN_EMAIL = 'd.feaux@easi.net'  # ← pas aan naar zijn echte email
DORIAN_COMPANY = 'EASI'          # ← zijn bedrijf

# 4a. Salesforce — Account voor Dorian's bedrijf (als nog niet aanwezig)
existing = sf_get_account_id(DORIAN_COMPANY)
if existing:
    DORIAN_ACCOUNT_ID = existing
    print(f'Account {DORIAN_COMPANY} bestaat al  id={DORIAN_ACCOUNT_ID}')
else:
    res = sf_post('Account', {
        'Name': DORIAN_COMPANY,
        'Industry': 'Technology',
        'BillingCity': 'Brussels',
        'BillingCountry': 'Belgium',
    })
    DORIAN_ACCOUNT_ID = res['id']
    print(f'Account {DORIAN_COMPANY} aangemaakt  id={DORIAN_ACCOUNT_ID}')

Account EASI bestaat al  id=001J6000002LijjIAC


In [26]:
# 4b. Salesforce — Contact: Dorian
res = sf_post('Contact', {
    'FirstName': 'Dorian',
    'LastName': DORIAN_LAST,
    'Email': DORIAN_EMAIL,
    'Title': 'Sales Engineer',
    'AccountId': DORIAN_ACCOUNT_ID,
})
print(f'Contact Dorian {DORIAN_LAST}  id={res["id"]}')

Contact Dorian Feaux  id=003KI000009uzRRYAY


In [27]:

# 4c. SmartSales — Locatie voor Dorian's bedrijf
res = ss_get_or_create({
    'name': f'{DORIAN_COMPANY} - Brussels',
    'code': f'{DORIAN_COMPANY[:4].upper()}-BRU-001',
    'city': 'Brussels',
    'country': 'Belgium',
    'street': 'Avenue Louise 65',
    'zip': '1050',
})


Locatie "EASI - Brussels" aangemaakt  uid=51f9e686-42da-11f1-a803-005056010707


---
## 5. Arne (persoon)

Nodig voor prompt: *Give me a complete overview of everything related to Arne*

> Arne is al vindbaar via `findpeople` in M365. Hier voegen we zijn SF-record + SS-locatie toe.

In [28]:
ARNE_LAST    = 'Albrecht'             # ← pas aan naar zijn echte achternaam
ARNE_EMAIL   = 'a.albrecht@easi.net' # ← pas aan naar zijn echte email
ARNE_COMPANY = 'EASI'            # ← zijn organisatie

# 5a. Salesforce — Account (hergebruik als al aanwezig)
existing = sf_get_account_id(ARNE_COMPANY)
if existing:
    ARNE_ACCOUNT_ID = existing
    print(f'Account {ARNE_COMPANY} bestaat al  id={ARNE_ACCOUNT_ID}')
else:
    res = sf_post('Account', {
        'Name': ARNE_COMPANY,
        'Industry': 'Technology',
        'BillingCity': 'Brussels',
        'BillingCountry': 'Belgium',
    })
    ARNE_ACCOUNT_ID = res['id']
    print(f'Account {ARNE_COMPANY} aangemaakt  id={ARNE_ACCOUNT_ID}')

Account EASI bestaat al  id=001J6000002LijjIAC


In [29]:
# 5b. Salesforce — Contact: Arne
res = sf_post('Contact', {
    'FirstName': 'Arne',
    'LastName': ARNE_LAST,
    'Email': ARNE_EMAIL,
    'Title': 'Business Analyst',
    'AccountId': ARNE_ACCOUNT_ID,
})
print(f'Contact Arne {ARNE_LAST}  id={res["id"]}')

Contact Arne Albrecht  id=003KI000009uzRbYAI


---
## 6. Ferrero / Nutella

Nodig voor prompt: *Find everything related to 'nutella' across all our systems*

In [30]:

# 6a. Salesforce — Account (hergebruik als al bestaat)
existing = sf_get_account_id('Ferrero Benelux')
if existing:
    FERRERO_ID = existing
    print(f'Account Ferrero Benelux bestaat al  id={FERRERO_ID}')
else:
    res = sf_post('Account', {
        'Name': 'Ferrero Benelux',
        'Industry': 'Food & Beverage',
        'BillingCity': 'Brussels',
        'BillingCountry': 'Belgium',
        'Description': 'Belgian subsidiary of Ferrero — produces Nutella, Kinder, Tic Tac.',
    })
    FERRERO_ID = res['id']
    print(f'Account Ferrero Benelux aangemaakt  id={FERRERO_ID}')


Account Ferrero Benelux aangemaakt  id=001KI00000N7zhFYAR


In [31]:
# 6b. Salesforce — Opportunity met 'Nutella' in naam
res = sf_post('Opportunity', {
    'Name': 'Ferrero — Nutella In-Store Display Campaign',
    'StageName': 'Prospecting',
    'CloseDate': '2026-10-01',
    'Amount': 30000,
    'AccountId': FERRERO_ID,
})
print(f'Opportunity  id={res["id"]}')

Opportunity  id=006KI000005bHlBYAU


In [32]:

# 6c. SmartSales — Locatie Ferrero/Nutella
res = ss_get_or_create({
    'name': 'Ferrero Benelux - Brussels (Nutella)',
    'code': 'FERR-BRU-001',
    'city': 'Brussels',
    'country': 'Belgium',
    'street': 'Bld de la Woluwe 42',
    'zip': '1200',
})


Locatie "Ferrero Benelux - Brussels (Nutella)" aangemaakt  uid=741d5f40-42da-11f1-a803-005056010707


---
## 7. Brussels (geografie)

Nodig voor prompt: *Give me a full picture of our commercial activity in Brussels*

> SmartSales heeft waarschijnlijk al Brusselse locaties. Hier voegen we een SF-account toe met BillingCity=Brussels dat ook in emails voorkomt.

In [33]:

# 7a. Salesforce — Account in Brussel (hergebruik als al bestaat)
existing = sf_get_account_id('Brussels Retail Partners')
if existing:
    print(f'Account Brussels Retail Partners bestaat al  id={existing}')
else:
    res = sf_post('Account', {
        'Name': 'Brussels Retail Partners',
        'Industry': 'Retail',
        'BillingCity': 'Brussels',
        'BillingCountry': 'Belgium',
        'BillingStreet': 'Grand Place 1',
        'BillingPostalCode': '1000',
    })
    print(f'Account Brussels Retail Partners aangemaakt  id={res["id"]}')


Account Brussels Retail Partners aangemaakt  id=001KI00000N7zSNYAZ


In [34]:
# 7b. SmartSales — extra Brusselse locatie (als nog geen Brussels locaties bestaan)
# Controleer eerst:
existing = ss_search('Brussels')
print(f'Bestaande Brussels-locaties: {len(existing)}')
for loc in existing[:5]:
    print(f'  - {loc.get("name")} ({loc.get("city")})')

Bestaande Brussels-locaties: 3
  - EASI - Brussels (Brussels)
  - Ferrero Benelux - Brussels (Nutella) (Brussels)
  - supplier1 - Brussels HQ (Brussels)


In [35]:
# Alleen uitvoeren als er GEEN Brussels-locaties zijn:
if len(existing) == 0:
    res = ss_post({
        'name': 'Brussels Retail Center',
        'code': 'BRU-RTC-001',
        'city': 'Brussels',
        'country': 'Belgium',
        'street': 'Rue Neuve 110',
        'zip': '1000',
    })
    print(f'SS Location aangemaakt  uid={res.get("uid")}')
else:
    print('Brussels-locaties al aanwezig — niets aangemaakt')

Brussels-locaties al aanwezig — niets aangemaakt


---
## 8. SmartSales — Catalog items & Orders

Voer de cellen hieronder **in volgorde** uit: eerst helpers, dan catalog items, dan locatie-UIDs ophalen, dan orders.

In [71]:

# Helper: catalog item get-or-create (client-side code filter — q param werkt niet op catalog/list)
def ss_get_or_create_catalog_item(payload: dict) -> dict:
    code = payload['code']
    url = f'{SS_BASE}/api/v3/catalog/list'
    r = httpx.get(url, params={'p': 'simple'}, headers={'Authorization': f'Bearer {SS_TOKEN}'})
    r.raise_for_status()
    entries = [e for e in r.json().get('entries', []) if e.get('code') == code]
    if entries:
        item = entries[0]
        print(f'↩ Catalog item "{item.get("title") or code}" bestaat al  uid={item.get("uid")}')
        return item
    r2 = httpx.post(f'{SS_BASE}/api/v3/catalog/item', json=payload,
                    headers={'Authorization': f'Bearer {SS_TOKEN}'})
    if not r2.is_success:
        print(f'❌ [{r2.status_code}]: {r2.text}')
        r2.raise_for_status()
    item = r2.json()
    print(f'✓ Catalog item "{item.get("title") or code}" aangemaakt  uid={item.get("uid")}')
    return item

# Helper: locatie ophalen op exacte naam
def ss_get_location(name: str) -> dict | None:
    import json as _json
    url = f'{SS_BASE}/api/v3/location/list'
    q = _json.dumps({'name': f'eq:{name}'})
    r = httpx.get(url, params={'q': q, 'p': 'simple'}, headers={'Authorization': f'Bearer {SS_TOKEN}'})
    r.raise_for_status()
    entries = r.json().get('entries', [])
    if not entries:
        print(f'⚠️  Locatie "{name}" niet gevonden')
        return None
    loc = entries[0]
    return {'uid': loc['uid'], 'code': loc.get('code', ''), 'name': loc.get('name', ''), 'externalId': None}

# Helper: order get-or-create op internalReference
def ss_get_or_create_order(ref: str, supplier: dict, customer: dict, items: list, comments: str = '') -> dict:
    from datetime import datetime, timezone
    import json as _json
    url = f'{SS_BASE}/api/v3/order/list'
    try:
        q = _json.dumps({'internalReference': f'eq:{ref}'})
        r = httpx.get(url, params={'q': q, 'p': 'simple'}, headers={'Authorization': f'Bearer {SS_TOKEN}'})
        if r.is_success:
            entries = r.json().get('entries', [])
            if entries:
                print(f'↩ Order "{ref}" bestaat al  uid={entries[0].get("uid")}')
                return entries[0]
    except Exception:
        pass

    date_str = datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S000+0000')
    order_items = []
    subtotal = 0.0
    for it in items:
        total = it['price'] * it['qty']
        subtotal += total
        order_items.append({
            'code': it['code'],
            'description': it['title'],
            'quantity': it['qty'],
            'price': it['price'],
            'totalPrice': total,
            'finalDiscountPrice': total,
            'discount': 0,
            'discountIsPercentage': False,
            'free': False,
            'hasAutoDiscounts': False,
            'autoDiscounts': [],
            'hasOverriddenAutoDiscount': False,
            'priceManuallySet': False,
        })
    payload = {
        'date': date_str,
        'internalReference': ref,
        'externalReference': ref,
        'type': 'ORDER',
        'approbationStatus': 'APPROVED',
        'comments': comments,
        'locale': 'en',
        'subtotal': subtotal,
        'total': subtotal,
        'totalQuantity': sum(it['qty'] for it in items),
        'supplier': supplier,
        'customer': customer,
        'items': order_items,
        'hasAutoDiscounts': False,
        'autoDiscounts': [],
        'totalModifiers': [],
        'offer': False,
        'discountVisible': False,
        'form': {},
        'customFields': {},
    }
    r2 = httpx.post(f'{SS_BASE}/api/v3/order', json=payload,
                    headers={'Authorization': f'Bearer {SS_TOKEN}'})
    if not r2.is_success:
        print(f'❌ [{r2.status_code}]: {r2.text}')
        r2.raise_for_status()
    order = r2.json()
    print(f'✓ Order "{ref}"  uid={order.get("uid")}  supplier={supplier["name"]}  customer={customer["name"]}')
    return order

print('✓ SS helpers geladen')


✓ SS helpers geladen


In [72]:
# Catalog items aanmaken — group is verplicht door de API
# Bestaande groups in het systeem (opgehaald via /api/v3/catalog/list):
GRP_HARDWARE = {'uid': 'a708e627-38ac-11f1-a803-005056010707', 'code': 'GRP-IT-HARDWARE', 'title': 'IT Hardware'}
GRP_SOFTWARE = {'uid': '881c615b-38ac-11f1-a803-005056010707', 'code': 'GRP-IT-SOFTWARE', 'title': 'IT Software'}
GRP_SPREADS  = {'uid': 'e900d857-2856-11f1-a2fe-005056010707', 'code': 'SPREADS',         'title': 'Spreads'}

CAT_SUPPLIER1 = ss_get_or_create_catalog_item({
    'code': 'PROD-SUP1-001',
    'title': 'supplier1 — Industrial Components Pack',
    'description': 'Standard component package delivered by supplier1. Used in manufacturing processes.',
    'price': 149.99,
    'salesUnit': 1,
    'unitOfMeasure': 'piece',
    'group': GRP_HARDWARE,
})

CAT_COLRUYT = ss_get_or_create_catalog_item({
    'code': 'PROD-COL-001',
    'title': 'Colruyt House Brand — Retail Display Kit',
    'description': 'In-store display kit for Colruyt Group retail locations.',
    'price': 89.50,
    'salesUnit': 1,
    'unitOfMeasure': 'piece',
    'group': GRP_HARDWARE,
})

CAT_NUTELLA = ss_get_or_create_catalog_item({
    'code': 'PROD-NUT-001',
    'title': 'Nutella 750g — Display Jar',
    'description': 'Nutella hazelnut spread 750g. Ferrero Benelux product for in-store promotion.',
    'price': 4.99,
    'salesUnit': 12,
    'unitOfMeasure': 'jar',
    'group': GRP_SPREADS,
})

CAT_GREENTECH = ss_get_or_create_catalog_item({
    'code': 'PROD-GTS-001',
    'title': 'GreenTech Solutions — Dashboard License',
    'description': 'Annual software license for the GreenTech sustainability dashboard.',
    'price': 2500.00,
    'salesUnit': 1,
    'unitOfMeasure': 'license',
    'group': GRP_SOFTWARE,
})

✓ Catalog item "supplier1 — Industrial Components Pack" aangemaakt  uid=6ed4f5e9-42e3-11f1-a803-005056010707
✓ Catalog item "Colruyt House Brand — Retail Display Kit" aangemaakt  uid=6fd1e538-42e3-11f1-a803-005056010707
✓ Catalog item "Nutella 750g — Display Jar" aangemaakt  uid=70b59307-42e3-11f1-a803-005056010707
✓ Catalog item "GreenTech Solutions — Dashboard License" aangemaakt  uid=718ccfb1-42e3-11f1-a803-005056010707


## get ids

In [73]:

# 10a. UIDs ophalen van de locaties die we nodig hebben voor orders
LOC_SUPPLIER1 = ss_get_location('supplier1')
LOC_CUSTOMER1 = ss_get_location('Customer1')   # bestaande testlocatie uit jemoeder.json
LOC_COLRUYT   = ss_get_location('Colruyt - Halle')
LOC_FERRERO   = ss_get_location('Ferrero Benelux - Brussels (Nutella)')
LOC_GREENTECH = ss_get_location('GreenTech Solutions - Ghent')

print()
for name, loc in [('supplier1', LOC_SUPPLIER1), ('Customer1', LOC_CUSTOMER1),
                   ('Colruyt', LOC_COLRUYT), ('Ferrero', LOC_FERRERO), ('GreenTech', LOC_GREENTECH)]:
    if loc:
        print(f'  {name:<20} uid={loc["uid"]}  code={loc["code"]}')



  supplier1            uid=cac5b20d-285c-11f1-a2fe-005056010707  code=SUP-001
  Customer1            uid=fa4acd29-285c-11f1-a2fe-005056010707  code=CUS-001
  Colruyt              uid=329a7e6c-42da-11f1-a803-005056010707  code=COLRUYT-HAL-001
  Ferrero              uid=741d5f40-42da-11f1-a803-005056010707  code=FERR-BRU-001
  GreenTech            uid=3d28b72d-42da-11f1-a803-005056010707  code=GTS-GNT-001


## effectief

In [74]:
# Orders aanmaken (idempotent via internalReference check)
if LOC_SUPPLIER1 and LOC_CUSTOMER1:
    ss_get_or_create_order(
        ref='ORD-SUP1-2026-001', supplier=LOC_SUPPLIER1, customer=LOC_CUSTOMER1,
        items=[{'code': 'PROD-SUP1-001', 'title': 'supplier1 — Industrial Components Pack', 'qty': 10, 'price': 149.99}],
        comments='Q2 2026 component delivery. Linked to open SF opportunity.',
    )

if LOC_COLRUYT and LOC_CUSTOMER1:
    ss_get_or_create_order(
        ref='ORD-COL-2026-001', supplier=LOC_COLRUYT, customer=LOC_CUSTOMER1,
        items=[{'code': 'PROD-COL-001', 'title': 'Colruyt House Brand — Retail Display Kit', 'qty': 5, 'price': 89.50}],
        comments='Retail display materials for Q3 rollout.',
    )

if LOC_FERRERO and LOC_CUSTOMER1:
    ss_get_or_create_order(
        ref='ORD-NUT-2026-001', supplier=LOC_FERRERO, customer=LOC_CUSTOMER1,
        items=[{'code': 'PROD-NUT-001', 'title': 'Nutella 750g — Display Jar', 'qty': 48, 'price': 4.99}],
        comments='Nutella in-store promotion stock for spring campaign.',
    )

if LOC_GREENTECH and LOC_CUSTOMER1:
    ss_get_or_create_order(
        ref='ORD-GTS-2026-001', supplier=LOC_GREENTECH, customer=LOC_CUSTOMER1,
        items=[{'code': 'PROD-GTS-001', 'title': 'GreenTech Solutions — Dashboard License', 'qty': 1, 'price': 2500.00}],
        comments='Annual sustainability dashboard license. Matches open SF opportunity.',
    )

✓ Order "ORD-SUP1-2026-001"  uid=76012bb6-42e3-11f1-a803-005056010707  supplier=supplier1  customer=Customer1
✓ Order "ORD-COL-2026-001"  uid=76e7ca8d-42e3-11f1-a803-005056010707  supplier=Colruyt - Halle  customer=Customer1
✓ Order "ORD-NUT-2026-001"  uid=77a50163-42e3-11f1-a803-005056010707  supplier=Ferrero Benelux - Brussels (Nutella)  customer=Customer1
✓ Order "ORD-GTS-2026-001"  uid=78655f28-42e3-11f1-a803-005056010707  supplier=GreenTech Solutions - Ghent  customer=Customer1


In [75]:

# Helper: catalog item get-or-create (op code)
def ss_get_or_create_catalog_item(payload: dict) -> dict:
    import json as _json
    code = payload['code']
    url = f'{SS_BASE}/api/v3/catalog/list'
    q = _json.dumps({'code': f'eq:{code}'})
    r = httpx.get(url, params={'q': q, 'p': 'simple'}, headers={'Authorization': f'Bearer {SS_TOKEN}'})
    r.raise_for_status()
    entries = r.json().get('entries', [])
    if entries:
        item = entries[0]
        print(f'Catalog item "{item.get("title") or code}" bestaat al  uid={item.get("uid")}')
        return item
    r2 = httpx.post(f'{SS_BASE}/api/v3/catalog/item', json=payload,
                    headers={'Authorization': f'Bearer {SS_TOKEN}'})
    if not r2.is_success:
        print(f'❌ [{r2.status_code}]: {r2.text}')
        r2.raise_for_status()
    item = r2.json()
    print(f'Catalog item "{item.get("title") or code}" aangemaakt  uid={item.get("uid")}')
    return item

# Helper: locatie ophalen op naam (geeft uid + code + name terug)
def ss_get_location(name: str) -> dict | None:
    import json as _json
    url = f'{SS_BASE}/api/v3/location/list'
    q = _json.dumps({'name': f'eq:{name}'})
    r = httpx.get(url, params={'q': q, 'p': 'simple'}, headers={'Authorization': f'Bearer {SS_TOKEN}'})
    r.raise_for_status()
    entries = r.json().get('entries', [])
    if not entries:
        print(f'⚠️ Locatie "{name}" niet gevonden')
        return None
    loc = entries[0]
    return {'uid': loc['uid'], 'code': loc.get('code', ''), 'name': loc.get('name', ''), 'externalId': None}

print('✓ catalog helpers geladen')


✓ catalog helpers geladen


---
## 10. SmartSales — Orders (locaties koppelen)

Orders die `supplier` en `customer` linken aan de locaties die we aangemaakt hebben. Zo zijn supplier1, Colruyt en Ferrero niet alleen locaties maar ook actief in orders.

---
## 9. SmartSales — Catalog items

Producten die gelinkt zijn aan onze test-entiteiten. Zo kan een query over "nutella" of "supplier1" ook catalog items teruggeven.

In [76]:
# Salesforce: overzicht van recent aangemaakte accounts
accounts = sf_query(
    "SELECT Id, Name, BillingCity, BillingCountry, Industry "
    "FROM Account "
    "WHERE Name IN ('supplier1','Colruyt Group','GreenTech Solutions','Ferrero Benelux','EASI','Brussels Retail Partners') "
    "ORDER BY Name"
)
print(f'=== Salesforce Accounts ({len(accounts)}) ===')
for a in accounts:
    print(f"  {a['Name']:<30} {a.get('BillingCity','?')}, {a.get('BillingCountry','?')}  [{a['Id']}]")

=== Salesforce Accounts (7) ===
  Brussels Retail Partners       Brussels, Belgium  [001KI00000N7zSNYAZ]
  Colruyt Group                  Halle, Belgium  [001KI00000N7zHSYAZ]
  EASI                           None, None  [001J6000002LijjIAC]
  EASI                           None, None  [001J6000009kz5ZIAQ]
  Ferrero Benelux                Brussels, Belgium  [001KI00000N7zhFYAR]
  GreenTech Solutions            Ghent, Belgium  [001KI00000N7zqXYAR]
  supplier1                      Brussels, Belgium  [001KI00000N7MjVYAV]


In [77]:
# Salesforce: contacts
contacts = sf_query(
    "SELECT FirstName, LastName, Email, Account.Name "
    "FROM Contact WHERE FirstName IN ('Marc','Dorian','Arne') ORDER BY FirstName"
)
print(f'=== Contacts ({len(contacts)}) ===')
for c in contacts:
    acc = (c.get('Account') or {}).get('Name', '?')
    print(f"  {c.get('FirstName','')} {c['LastName']:<20} {c.get('Email','?'):<35} [{acc}]")

=== Contacts (5) ===
  Arne Albrecht             albrecht.arne@gmail.com             [Displaytech]
  Arne Albrecht             a.albrecht@easi.net                 [EASI]
  Dorian Feaux                d.feaux@easi.net                    [EASI]
  Marc Benioff              info@salesforce.com                 [salesforce.com]
  Marc Supplier             marc.supplier@supplier1.be          [supplier1]


---
## 9. Microsoft 365 — Emails, Calendar events & OneDrive

Voer onderstaande cellen in volgorde uit:
1. **Graph auth** — device code login (browser popup)
2. **Graph helpers** — hulpfuncties laden
3. **Emails** — 7 emails naar jezelf sturen (idempotent op subject)
4. **Calendar events** — 5 afspraken aanmaken (idempotent op subject)
5. **OneDrive** — supplier1 rapport uploaden

In [4]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import configparser, httpx
from startup import authenticate

config = configparser.ConfigParser()
config.read('../config.cfg')
az = config['azure']

scopes = az['graphUserScopes'].split()
GRAPH_TOKEN = authenticate(
    client_id=az['clientId'],
    tenant_id=az['tenantId'],
    scopes=scopes,
    client_secret=az['clientSecret'],
)

# Eigen email ophalen via raw httpx (geen asyncio nodig)
r = httpx.get('https://graph.microsoft.com/v1.0/me',
               params={'$select': 'mail,userPrincipalName'},
               headers={'Authorization': f'Bearer {GRAPH_TOKEN}'})
r.raise_for_status()
me = r.json()
MY_EMAIL = me.get('mail') or me.get('userPrincipalName')
print(f'✓ Graph auth  user={MY_EMAIL}')

c:\Users\AALB\Desktop\mp\graph\graphxmaf\.venv\Lib\site-packages\msal\oauth2cli\oauth2.py:407: UserWarning: response_mode='form_post' is recommended for better security. See https://www.rfc-editor.org/rfc/rfc9700.html#section-4.3.1
  warnings.warn(



Opening browser for login...
Waiting for authentication callback...
✓ Graph auth  user=SmartAdmin@easigroupdemo.onmicrosoft.com


In [5]:
# Graph helpers — emails, calendar events, OneDrive
GRAPH_BASE = 'https://graph.microsoft.com/v1.0'

def _graph_headers():
    return {'Authorization': f'Bearer {GRAPH_TOKEN}', 'Content-Type': 'application/json'}

def graph_create_inbox_message_idempotent(subject: str, body: str,
                                           from_address: str, from_name: str) -> None:
    """Maak een email direct in inbox aan met custom afzender — geen echte SMTP-verzending."""
    r = httpx.get(f'{GRAPH_BASE}/me/mailFolders/inbox/messages',
                  params={'$filter': f"subject eq '{subject}'", '$top': '1', '$select': 'id'},
                  headers={'Authorization': f'Bearer {GRAPH_TOKEN}'})
    if r.is_success and r.json().get('value'):
        print(f'↩ Inbox email al aanwezig: "{subject}"')
        return
    payload = {
        'subject': subject,
        'body': {'contentType': 'Text', 'content': body},
        'from':        {'emailAddress': {'address': from_address, 'name': from_name}},
        'sender':      {'emailAddress': {'address': from_address, 'name': from_name}},
        'toRecipients': [{'emailAddress': {'address': MY_EMAIL}}],
        'isDraft': False,
        'isRead': False,
    }
    r2 = httpx.post(f'{GRAPH_BASE}/me/mailFolders/inbox/messages', json=payload,
                    headers={'Authorization': f'Bearer {GRAPH_TOKEN}'})
    if not r2.is_success:
        print(f'❌ [{r2.status_code}]: {r2.text[:200]}')
        r2.raise_for_status()
    print(f'✓ Inbox email aangemaakt: "{subject}"  (van: {from_name} <{from_address}>)')

def graph_create_event_idempotent(subject: str, body: str, start: str, end: str,
                                   attendees: list[str] = []) -> None:
    """Maak calendar event aan als subject nog niet bestaat."""
    r = httpx.get(f'{GRAPH_BASE}/me/events',
                  params={'$filter': f"subject eq '{subject}'", '$top': '1', '$select': 'id'},
                  headers={'Authorization': f'Bearer {GRAPH_TOKEN}'})
    if r.is_success and r.json().get('value'):
        print(f'↩ Event al aanwezig: "{subject}"')
        return
    payload = {
        'subject': subject,
        'body': {'contentType': 'Text', 'content': body},
        'start': {'dateTime': start, 'timeZone': 'Europe/Brussels'},
        'end':   {'dateTime': end,   'timeZone': 'Europe/Brussels'},
        'attendees': [{'emailAddress': {'address': a}, 'type': 'required'} for a in attendees],
    }
    r2 = httpx.post(f'{GRAPH_BASE}/me/events', json=payload,
                    headers={'Authorization': f'Bearer {GRAPH_TOKEN}'})
    if not r2.is_success:
        print(f'❌ [{r2.status_code}]: {r2.text[:200]}')
        r2.raise_for_status()
    print(f'✓ Event aangemaakt: "{subject}"')

def graph_upload_file(filename: str, content: str) -> None:
    """Upload tekstbestand naar OneDrive root (overschrijft als al bestaat)."""
    r = httpx.put(f'{GRAPH_BASE}/me/drive/root:/{filename}:/content',
                  content=content.encode('utf-8'),
                  headers={'Authorization': f'Bearer {GRAPH_TOKEN}', 'Content-Type': 'text/plain'})
    if not r.is_success:
        print(f'❌ [{r.status_code}]: {r.text[:200]}')
        r.raise_for_status()
    print(f'✓ Bestand geupload: "{filename}"')

print('✓ Graph helpers geladen')

✓ Graph helpers geladen


In [6]:
graph_create_inbox_message_idempotent(
    subject='Follow-up: supplier1 Q2 2026 Delivery Contract',
    body="""Hi,

Following up on the Q2 2026 delivery contract for supplier1.
The April batch arrived 5 days late — we are investigating the root cause at our Brussels site.

Please confirm the new delivery schedule by end of week.

Best regards,
Marc Supplier
supplier1 — Rue du Commerce 1, Brussels""",
    from_address='marc.supplier@supplier1.be',
    from_name='Marc Supplier',
)

❌ [403]: {"error":{"code":"ErrorAccessDenied","message":"Access is denied. Check credentials and try again."}}


HTTPStatusError: Client error '403 Forbidden' for url 'https://graph.microsoft.com/v1.0/me/mailFolders/inbox/messages'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/403

In [ ]:
# Emails aanmaken in inbox met correcte afzenders (geen echte SMTP-verzending)

# supplier1 — 2 emails van Marc Supplier
graph_create_inbox_message_idempotent(
    subject='Follow-up: supplier1 Q2 2026 Delivery Contract',
    body="""Hi,

Following up on the Q2 2026 delivery contract for supplier1.
The April batch arrived 5 days late — we are investigating the root cause at our Brussels site.

Please confirm the new delivery schedule by end of week.

Best regards,
Marc Supplier
supplier1 — Rue du Commerce 1, Brussels""",
    from_address='marc.supplier@supplier1.be',
    from_name='Marc Supplier',
)

graph_create_inbox_message_idempotent(
    subject='supplier1 — Updated pricing for H2 2026',
    body="""Hi,

Please find attached the updated pricing sheet from supplier1 for the second half of 2026.
The Industrial Components Pack (PROD-SUP1-001) will see a 3% price adjustment from July 1st.

Let me know if you have any questions.

Marc Supplier
Account Manager — supplier1""",
    from_address='marc.supplier@supplier1.be',
    from_name='Marc Supplier',
)

# Colruyt — 1 email van Colruyt contactpersoon
graph_create_inbox_message_idempotent(
    subject='Colruyt Group — SmartSales pilot bevestiging',
    body="""Goedemiddag,

Hierbij bevestigen wij vanuit Colruyt Group de deelname aan de SmartSales piloot in onze Halle-vestiging.
Startdatum: juni 2026. Contactpersoon onze kant: procurement team Halle.

Met vriendelijke groeten,
Colruyt Group Procurement""",
    from_address='procurement@colruyt.be',
    from_name='Colruyt Group Procurement',
)

# GreenTech Solutions — 1 email
graph_create_inbox_message_idempotent(
    subject='GreenTech Solutions — Interest in sustainability dashboard',
    body="""Hi,

We at GreenTech Solutions (Ghent) are interested in the annual sustainability dashboard license.
Could you send us a formal proposal? Our budget is around EUR 2,500/year.

Kind regards,
GreenTech Solutions — Technology Team
Technologiepark 122, Ghent""",
    from_address='info@greentechsolutions.be',
    from_name='GreenTech Solutions',
)

# Ferrero / Nutella — 1 email
graph_create_inbox_message_idempotent(
    subject='Ferrero Benelux — Nutella spring campaign inquiry',
    body="""Hello,

Ferrero Benelux would like to discuss in-store display options for our Nutella spring campaign.
We are looking for display jars (750g format) for our Brussels-area retail partners.

Please contact us to schedule a call.

Ferrero Benelux — Marketing
Bld de la Woluwe 42, Brussels""",
    from_address='marketing@ferrero.be',
    from_name='Ferrero Benelux Marketing',
)

# Dorian — 1 email van DORIAN_EMAIL
graph_create_inbox_message_idempotent(
    subject=f'Intro: Dorian Feaux — EASI collaboration proposal',
    body=f"""Hi,

I'm Dorian Feaux, Sales Engineer at EASI (Avenue Louise 65, Brussels).
I'd like to discuss a potential collaboration on the Brussels-area projects you mentioned.

Looking forward to connecting.

Dorian Feaux
EASI — {DORIAN_EMAIL}""",
    from_address=DORIAN_EMAIL,
    from_name='Dorian Feaux',
)

# Belgium + Brussels — 1 email van intern/collega
graph_create_inbox_message_idempotent(
    subject='Belgium region — Q1 2026 commercial summary',
    body=f"""Hi,

Quick summary of our Belgium pipeline for Q1 2026:

Active accounts: supplier1 (Brussels), Colruyt Group (Halle), GreenTech Solutions (Ghent),
Ferrero Benelux (Brussels), Brussels Retail Partners (Brussels).

Total open pipeline Belgium: EUR 375,000.
Brussels remains our most active city with 3 active accounts.

Key contacts: Marc Supplier (supplier1), Dorian Feaux (EASI, Brussels), procurement@colruyt.be.

Regards""",
    from_address=MY_EMAIL,
    from_name='My Team',
)

In [ ]:
# Calendar events aanmaken (idempotent op subject)
# Geen attendees — alleen jouw eigen kalender, niemand krijgt een invite
graph_create_event_idempotent(
    subject='supplier1 — Q2 Delivery Review',
    body='Review of the Q2 2026 delivery contract with supplier1 (marc.supplier@supplier1.be). Discuss the April delay and new schedule.',
    start='2026-05-10T10:00:00', end='2026-05-10T11:00:00',
)
graph_create_event_idempotent(
    subject='Colruyt Group — SmartSales Rollout Kickoff',
    body='Kickoff meeting for the SmartSales rollout at Colruyt Group. Pilot site: Colruyt - Halle.',
    start='2026-05-15T14:00:00', end='2026-05-15T15:30:00',
)
graph_create_event_idempotent(
    subject='GreenTech Solutions — Technical Qualification Call',
    body='Technical qualification call with GreenTech Solutions for the sustainability dashboard license.',
    start='2026-05-12T11:00:00', end='2026-05-12T12:00:00',
)
graph_create_event_idempotent(
    subject=f'Weekly sync — Dorian Feaux',
    body=f'Weekly sync with Dorian Feaux ({DORIAN_EMAIL}, EASI) to discuss ongoing projects and Brussels activity.',
    start='2026-05-06T09:00:00', end='2026-05-06T09:30:00',
)
graph_create_event_idempotent(
    subject='Sprint review — Arne Albrecht',
    body=f'Sprint review with Arne Albrecht ({ARNE_EMAIL}) to go over agent evaluation results.',
    start='2026-05-08T14:00:00', end='2026-05-08T15:00:00',
)

In [ ]:
# OneDrive — supplier1 rapport uploaden (PUT overschrijft als al bestaat)
graph_upload_file(
    'supplier1_delivery_report_Q2_2026.txt',
    """supplier1 - Q2 2026 Delivery Report
=====================================

Supplier: supplier1 (SUP-001)
Period: Q2 2026 (April-June)

Orders:
- ORD-SUP1-2026-001: 10x Industrial Components Pack @ EUR 149.99 = EUR 1499.90
  Status: Shipped (5 days delay - under investigation)

Open Issues:
- April batch arrived late. supplier1 quality review initiated.
- Contract renewal (supplier1 Q2 Delivery Contract) pending signature.

Contact: Marc Supplier <marc.supplier@supplier1.be>
Salesforce Account ID: 001KI00000N7MjVYAV
SmartSales Location UID: cac5b20d-285c-11f1-a2fe-005056010707
"""
)